# Imports

In [ ]:
%load_ext autoreload
%autoreload 2

from transformers import RobertaForSequenceClassification
import torch
from torch.optim import AdamW
from transformers import get_scheduler, Trainer, TrainingArguments
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import evaluate
from peft import LoraConfig, TaskType
from peft import get_peft_model
import huggingface_hub

# Load The Model

In [ ]:
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules="all-linear"
)

id2label = {0: "Bearish", 1: "Bullish"}
label2id = {"Bearish": 0, "Bullish": 1}

model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2, id2label=id2label, label2id=label2id)
tokenizer = AutoTokenizer.from_pretrained('roberta-base')

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Prepare Dataset

In [ ]:
from datasets import Dataset
from transformers import DataCollatorWithPadding

pm_df = pd.read_parquet('../valid_comment_sentiment.parquet')

train_df, val_df = train_test_split(pm_df, test_size=0.2, random_state=42, stratify=pm_df['label'])

train_dataset = Dataset.from_pandas(train_df[['body', 'label']])
test_dataset = Dataset.from_pandas(val_df[['body', 'label']])

def tokenize_function(examples):
    return tokenizer(examples['body'], truncation=True, padding='max_length', max_length=256)

tokenized_datasets = {
    'train': train_dataset.map(tokenize_function, batched=True),
    'test': test_dataset.map(tokenize_function, batched=True)
}

tokenized_datasets['train'] = tokenized_datasets['train'].rename_column('label', 'labels')
tokenized_datasets['test'] = tokenized_datasets['test'].rename_column('label', 'labels')

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    metric = evaluate.combine(["accuracy", "f1", "precision", "recall"])
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

subset_train = tokenized_datasets['train'].shuffle(seed=42).select(range(1000))
subset_test = tokenized_datasets['test'].shuffle(seed=42).select(range(200))

In [7]:
import pandas as pd
pm_df = pd.read_parquet('../valid_comment_sentiment.parquet')
pm_df.to_csv('../valid_comment_sentiment.csv')

# Use model for inference

In [ ]:
from peft import AutoPeftModelForSequenceClassification
from transformers import AutoTokenizer

output_dir = "./roberta-lora-sentiment"
model = AutoPeftModelForSequenceClassification.from_pretrained(output_dir)
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

model = model.to("cpu")
model.eval()
inputs = tokenizer("Preheat the oven to 350 degrees and place the cookie dough", return_tensors="pt")

model(**inputs)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


SequenceClassifierOutput(loss=None, logits=tensor([[ 0.1515, -0.2945]]), hidden_states=None, attentions=None)